# Face Detection and Landmarks

> **Advanced · Applied vision**


## Why this matters

Face detection is a useful case study in the evolution from handcrafted cascades to neural detectors; landmarks add geometry for alignment and analysis.

**Where it appears:** Photo organization with consent, camera framing, accessibility tooling, and landmark-driven interaction prototypes.


## Learning Objectives

- Run pre-trained Haar cascade classifiers correctly, tuning key parameters
- Understand why Haar cascades need multi-scale search and what each parameter controls
- Post-process raw detections with NMS-style overlap filtering
- Detect faces with OpenCV's DNN face detector (more robust than Haar cascades)
- Locate facial landmarks and understand what downstream tasks they enable
- Build a face-alignment function using landmark-based rotation correction


## Prerequisites

13 Video Processing and Background Motion; 21 DNN and ONNX Inference is recommended for the neural-detector portion

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

Haar `detectMultiScale`, DNN face detection, landmark geometry, affine alignment, non-maximum suppression

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Haar Cascades and Classical Detection

Haar cascades (Viola-Jones, 2001) are a classical, fast, CPU-friendly
object detector using cascades of simple rectangular features. They are
largely superseded by deep learning for accuracy, but remain useful for
lightweight/embedded scenarios. `detectMultiScale`'s parameters
(`scaleFactor`, `minNeighbors`, `minSize`) directly trade off speed,
false-positive rate, and recall -- tuning them deliberately (not leaving
defaults) is the actual skill here.


### Face Detection and Facial Landmarks

Modern face detection uses a small DNN (e.g. an SSD-based res10 model
shipped with OpenCV samples) rather than Haar cascades, for much better
robustness to pose/lighting. Facial landmarks (eye corners, nose tip,
mouth corners) enable **face alignment** -- rotating/scaling a detected
face crop so the eyes are level and at consistent positions, which is a
required pre-processing step for face recognition (next notebook).


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Haar Cascades and Classical Detection


### 1. Loading a bundled cascade and running detection

OpenCV ships pre-trained cascades inside `cv2.data.haarcascades`. Load the frontal face cascade and run it on a synthetic face-like image.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid

cascade_path = get_real_data("models", "haarcascade_frontalface_default.xml")
face_cascade = cv2.CascadeClassifier(cascade_path)
assert not face_cascade.empty(), (
    "Failed to load Haar cascade -- check your OpenCV install"
)

_face = load_real_image("images/faces", "face.jpg")
gray = cv2.cvtColor(_face, cv2.COLOR_BGR2GRAY)

detections = face_cascade.detectMultiScale(
    gray, scaleFactor=1.05, minNeighbors=4, minSize=(60, 60)
)
print(f"Detections on  face-like shape: {len(detections)}")
print("(Note: Haar cascades are trained on real faces, so a stylized  shape may or")
print(
    " may not trigger a detection -- this cell demonstrates the API regardless of outcome.)"
)

### 2. Understanding detectMultiScale's parameters

`scaleFactor` controls how much the search window shrinks between scales (smaller = more thorough but slower); `minNeighbors` controls how many overlapping candidate detections are required to keep one (higher = fewer false positives, but might drop true positives).


In [ ]:
def compare_parameters(gray: np.ndarray, cascade) -> None:
    configs = [
        dict(scaleFactor=1.05, minNeighbors=3, label="thorough, permissive"),
        dict(scaleFactor=1.3, minNeighbors=8, label="fast, strict"),
    ]
    for cfg in configs:
        dets = cascade.detectMultiScale(
            gray,
            scaleFactor=cfg["scaleFactor"],
            minNeighbors=cfg["minNeighbors"],
            minSize=(40, 40),
        )
        print(
            f"{cfg['label']:22s} (scaleFactor={cfg['scaleFactor']}, minNeighbors={cfg['minNeighbors']}) "
            f"-> {len(dets)} detections"
        )


compare_parameters(gray, face_cascade)

### 3. A robust wrapper with overlap filtering

Wrap cascade detection with `cv2.dnn.NMSBoxes` to remove near-duplicate boxes -- the same overlap-suppression concern as template matching, applied here to classical detection.


In [ ]:
def detect_and_draw(image: np.ndarray, cascade, **kwargs) -> np.ndarray:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    boxes = cascade.detectMultiScale(gray, **kwargs)
    if len(boxes) == 0:
        return image.copy()

    rects = [[int(x), int(y), int(w), int(h)] for x, y, w, h in boxes]
    scores = [1.0] * len(rects)
    keep = cv2.dnn.NMSBoxes(rects, scores, score_threshold=0.0, nms_threshold=0.3)
    keep = np.array(keep).flatten() if len(keep) else []

    annotated = image.copy()
    for i in keep:
        x, y, w, h = rects[i]
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
    return annotated


result = detect_and_draw(
    load_real_image("images/faces", "face.jpg"),
    face_cascade,
    scaleFactor=1.1,
    minNeighbors=4,
    minSize=(50, 50),
)
show_grid(
    [
        ("real input", load_real_image("images/faces", "face.jpg")),
        ("detections (NMS applied)", result),
    ]
)

## Part 2: Face Detection and Facial Landmarks


### 1. Detecting faces (with a graceful fallback)

Prefer OpenCV's DNN face detector; if the model files aren't available in this environment, fall back to the Haar cascade so the notebook still runs end-to-end.



In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid, has_module


def detect_faces(image: np.ndarray) -> list:
    """Detect faces, preferring Haar cascade here since it needs no extra downloaded weights.
    In production, swap in cv2.dnn.readNetFromTensorflow(...) with the res10 SSD face model for
    substantially better accuracy -- see notebook 34 (OpenCV DNN Fundamentals) for that pattern."""
    cascade = cv2.CascadeClassifier(
        get_real_data("models", "haarcascade_frontalface_default.xml")
    )
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    boxes = cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=4, minSize=(50, 50)
    )
    return [(int(x), int(y), int(w), int(h)) for x, y, w, h in boxes]


_face = load_real_image("images/faces", "face.jpg")
faces = detect_faces(_face)
print(f"Detected {len(faces)} face-like region(s)")

annotated = _face.copy()
for x, y, w, h in faces:
    cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
show_grid([("input", _face), ("detected", annotated)])

### 2. Approximate landmark localization

Without a trained landmark model available offline, approximate eye/nose/mouth positions geometrically within the detected face box -- illustrating what a real landmark model (e.g. dlib's 68-point, or `cv2.face.createFacemarkLBF`) outputs, and how alignment logic consumes those points.



In [ ]:
def approximate_landmarks(face_box: tuple) -> dict:
    """Standard relative landmark positions within a face box (rough, illustrative).
    A real facemark model returns learned positions instead of these fixed ratios."""
    x, y, w, h = face_box
    return {
        "left_eye": (x + int(0.30 * w), y + int(0.40 * h)),
        "right_eye": (x + int(0.70 * w), y + int(0.40 * h)),
        "nose_tip": (x + int(0.50 * w), y + int(0.55 * h)),
        "mouth_left": (x + int(0.35 * w), y + int(0.75 * h)),
        "mouth_right": (x + int(0.65 * w), y + int(0.75 * h)),
    }


img = load_real_image("images/faces", "face.jpg")
face_boxes = detect_faces(img)
if len(face_boxes) > 0:
    lms = approximate_landmarks(face_boxes[0])
    annotated = img.copy()
    for name, pt in lms.items():
        cv2.circle(annotated, pt, 4, (0, 0, 255), -1)
    show_grid([("Approximate Landmarks on Real Face", annotated)])

### 3. Face alignment using landmark geometry

Rotate the face so a line through both eyes is horizontal -- the standard pre-processing step before face recognition embeddings are computed.



In [ ]:
def align_face(image: np.ndarray, left_eye: tuple, right_eye: tuple) -> np.ndarray:
    """Rotate `image` so the line between the two eye points becomes horizontal."""
    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]
    angle = np.degrees(np.arctan2(dy, dx))
    center = ((left_eye[0] + right_eye[0]) // 2, (left_eye[1] + right_eye[1]) // 2)
    M = cv2.getRotationMatrix2D(center, angle, scale=1.0)
    return cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))


tilted_face = cv2.warpAffine(
    load_real_image("images/faces", "face.jpg"),
    cv2.getRotationMatrix2D((160, 160), 20, 1.0),
    (320, 320),
)
tilted_landmarks = approximate_landmarks((40, 60, 220, 220))  # approx box after tilt
aligned = align_face(
    tilted_face, tilted_landmarks["left_eye"], tilted_landmarks["right_eye"]
)

show_grid([("tilted face", tilted_face), ("aligned (eyes leveled)", aligned)])

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Haar Cascades and Classical Detection: Non-Maximum Suppression (NMS) for Haar Detections

Haar cascade detectors often produce overlapping bounding boxes around the same target. To clean these duplicates, we implement Non-Maximum Suppression (NMS) which groups bounding boxes based on Intersection-over-Union (IoU) scores.


In [ ]:
# Create mock overlapping detections around a face: format (x, y, w, h)
detections = [
    [100, 100, 80, 80],
    [105, 102, 78, 82],
    [98, 97, 85, 80],
    [220, 50, 60, 60],  # Another distinct object box
]


def nms_bboxes(boxes: list[list[int]], overlap_thresh: float = 0.3) -> list[list[int]]:
    if len(boxes) == 0:
        return []

    boxes_arr = np.array(boxes, dtype=np.float32)
    x1 = boxes_arr[:, 0]
    y1 = boxes_arr[:, 1]
    x2 = x1 + boxes_arr[:, 2]
    y2 = y1 + boxes_arr[:, 3]

    # Calculate areas
    areas = (x2 - x1) * (y2 - y1)
    idxs = np.argsort(y2)

    pick = []
    while len(idxs) > 0:
        last = len(idxs) - 1
        i = idxs[last]
        pick.append(i)

        # Calculate overlaps
        xx1 = np.maximum(x1[i], x1[idxs[:last]])
        yy1 = np.maximum(y1[i], y1[idxs[:last]])
        xx2 = np.minimum(x2[i], x2[idxs[:last]])
        yy2 = np.minimum(y2[i], y2[idxs[:last]])

        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)

        overlap = (w * h) / areas[idxs[:last]]
        idxs = np.delete(
            idxs, np.concatenate(([last], np.where(overlap > overlap_thresh)[0]))
        )

    return boxes_arr[pick].astype(np.int32).tolist()


pruned = nms_bboxes(detections)
print("Input boxes count: 4 | NMS remaining boxes count:", len(pruned))
print("Final detections:", pruned)

### Mini Project — Face Detection and Facial Landmarks: Eye Aspect Ratio (EAR) for Drowsiness Detection

A classic application of face landmarks is measuring fatigue by computing the Eye Aspect Ratio (EAR). By measuring the ratio of vertical distance between eye contour landmarks to the horizontal distance, we can determine whether the eyes are open or closed.



In [ ]:
def calculate_ear(eye_landmarks: np.ndarray) -> float:
    # eye_landmarks coordinates: [p1, p2, p3, p4, p5, p6] (standard format)
    # Vertical distances
    a = np.linalg.norm(eye_landmarks[1] - eye_landmarks[5])
    b = np.linalg.norm(eye_landmarks[2] - eye_landmarks[4])
    # Horizontal distance
    c = np.linalg.norm(eye_landmarks[0] - eye_landmarks[3])

    # EAR formula
    return (a + b) / (2.0 * c)


# Mock eye landmarks (6 points)
# 1. Open eye coordinates
open_eye = np.array(
    [[10, 20], [15, 12], [25, 12], [30, 20], [25, 28], [15, 28]], dtype=np.float32
)
# 2. Closed eye coordinates (collapsed vertically)
closed_eye = np.array(
    [[10, 20], [15, 19], [25, 19], [30, 20], [25, 21], [15, 21]], dtype=np.float32
)

print(f"Open eye EAR score: {calculate_ear(open_eye):.4f}")
print(f"Closed eye EAR score: {calculate_ear(closed_eye):.4f}")

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Haar Cascades and Classical Detection
1. Load `haarcascade_eye.xml` and run it restricted to the upper half of a detected face region only.
2. Benchmark `detectMultiScale` runtime for `scaleFactor` in [1.05, 1.1, 1.3] using `cv_utils.Timer`.
3. List, in a markdown cell, three real-world scenarios where a Haar cascade is still preferable to a deep-learning detector, and why.

Use the empty cell below to work through them.


#### Solutions — Haar Cascades and Classical Detection

In [ ]:
# Solution 1: restrict eye cascade to upper half of face
def detect_eyes_in_face(
    face_image: np.ndarray, face_bbox: tuple[int, int, int, int], eye_cascade
) -> list[tuple[int, int, int, int]]:
    """Restrict eye search to upper 50% of the bounding box to reduce false hits."""
    fx, fy, fw, fh = face_bbox
    # Extract upper face region of interest
    upper_face_roi = face_image[fy : fy + fh // 2, fx : fx + fw]

    eyes = eye_cascade.detectMultiScale(upper_face_roi)
    # Translate eye coordinates back to global canvas space
    global_eyes = []
    for ex, ey, ew, eh in eyes:
        global_eyes.append((fx + ex, fy + ey, ew, eh))
    return global_eyes

In [ ]:
# Solution 2: detectMultiScale scaleFactor speed sweep
# Explanation: `scaleFactor` controls the scale step factor during search. A low value (1.05)
# checks many intermediate sizes, yielding high accuracy but runs slow. A high value (1.3)
# skips sizes, increasing search speed (2-3x faster) but missing smaller objects.


In [ ]:
# Solution 3: Scenarios where Haar is preferred
# 1. Resource-constrained microcontrollers / legacy embedded systems without neural hardware.
# 2. Hard real-time applications requiring low latency (sub-5ms) face tracking on CPU.
# 3. Simple binary setups where target objects are standard and highly rigid (e.g. barcode location).


### Exercises — Face Detection and Facial Landmarks
1. Replace the Haar-based `detect_faces` with `cv2.dnn.readNetFromTensorFlow` and the res10 SSD model if you can download model weights in your environment.
2. Extend `align_face` to also crop tightly around the aligned face using the landmark bounding box.
3. Write a function that estimates head tilt angle directly from two eye landmark points, independent of alignment.

Use the empty cell below to work through them.



#### Solutions — Face Detection and Facial Landmarks

In [ ]:
def run_dnn_face(img):
    net = cv2.dnn.readNetFromTensorflow(
        get_real_data("models", "opencv_face_detector_uint8.pb"),
        get_real_data("models", "opencv_face_detector.pbtxt"),
    )
    blob = cv2.dnn.blobFromImage(img, 1.0, (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    dets = net.forward()
    annotated = img.copy()
    h, w = img.shape[:2]
    for i in range(dets.shape[2]):
        confidence = dets[0, 0, i, 2]
        if confidence > 0.5:
            box = dets[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")
            cv2.rectangle(annotated, (startX, startY), (endX, endY), (0, 255, 0), 2)
    return annotated


img = load_real_image("images/faces", "group.jpg")
res = run_dnn_face(img)
show_grid([("DNN Face Detection", res)])

In [ ]:
# Solution 2: Crop tightly around face using landmark bounding box
def crop_aligned_face(image: np.ndarray, landmarks: np.ndarray) -> np.ndarray:
    """Find landmarks envelope coordinates and crop."""
    x, y, w, h = cv2.boundingRect(landmarks)
    # Add margin padding
    margin = int(0.1 * w)
    lh, lw = image.shape[:2]
    return image[
        max(0, y - margin) : min(lh, y + h + margin),
        max(0, x - margin) : min(lw, x + w + margin),
    ]


img = load_real_image("images/faces", "face.jpg")
boxes = detect_faces(img)
if boxes:
    lms = approximate_landmarks(boxes[0])
    pts = np.array(list(lms.values()))
    cropped = crop_aligned_face(img, pts)
    show_grid([("Tight Face Crop", cropped)])

In [ ]:
# Solution 3: Estimate head tilt angle directly from eye points
def estimate_head_tilt(left_eye: tuple[int, int], right_eye: tuple[int, int]) -> float:
    """Estimate angle of head tilt in degrees based on slope of eyes."""
    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    return np.degrees(np.arctan2(dy, dx))

## Summary

You can compare legacy and DNN-based detection, align a detected face, and explain sensitivity to pose, lighting, demographic variation, and image quality.

- **Best Practices:** Obtain consent, test across representative conditions, keep face data local where possible, and show uncertainty or failure rather than overclaiming accuracy.
- **Common Pitfalls:** Treating a Haar cascade as robust detection, using approximate landmarks for sensitive decisions, and collecting biometric data without a clear purpose or consent.